In [12]:
#!/usr/bin/env python3
import csv
import math
import sys

import actionlib
import rospy
from moveit_commander import (MoveGroupCommander, PlanningSceneInterface,
                              RobotCommander, RobotState, roscpp_initialize,
                              roscpp_shutdown)
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray
from telehandler_moveit.msg import (MoveToPositionAction, MoveToPositionGoal,
                                    MoveToPositionResult)

joint_state = None
low_level_joints = None


def euclidean_distance(p1, p2):
    return math.sqrt(sum([(a - b) ** 2 for a, b in zip(p1, p2)]))


def state_cb(msg):

    global joint_state
    joint_state = [msg.position[0], msg.position[1] + msg.position[2] +
                   msg.position[3] + 6.390909194946289, msg.position[5]]
    # Process the RobotState message
    # rospy.loginfo(f"Received Positions {[msg.position[0], msg.position[2]+ (msg.position[3]+ msg.position[4])*10, msg.position[5]]}")


def low_level_joints_cb(msg):
    global low_level_joints
    low_level_joints = msg.data
    # Process the Float32MultiArray message
    # rospy.loginfo(f"Received low level joints: {msg.data}")
    
def roscpp_shutdown():
    rospy.loginfo("Shutting down MoveIt Commander...")
    move_group = MoveGroupCommander("Arm_Group",
                                    robot_description="/ugv0/telehandler/robot_description",
                                    ns="/ugv0/telehandler")
    move_group.stop()
    move_group.clear_pose_targets()
    rospy.loginfo("MoveIt Commander shutdown complete.")


def main():
    rospy.loginfo("Initializing MoveIt Commander...")

    move_group = MoveGroupCommander("Arm_Group",
                                    robot_description="/ugv0/telehandler/robot_description",
                                    ns="/ugv0/telehandler",
                                    wait_for_servers=5)

    desired_positions = [
        [7.3, 0.0, 1.5],
        [8.0, 0.0, 2.4],
        [9.0, -2.0, 3.6],
        [8.0, -1.5, 2.4],
        [7.3, -1.0, 1.5]
    ]

    _sub = rospy.Subscriber(
        "/ugv0/telehandler/joint_states", JointState, state_cb)
    _low_level_sub = rospy.Subscriber(
        "/ugv0/telehandler/joints", Float32MultiArray, low_level_joints_cb)

    _action_goal = actionlib.SimpleActionClient(
        '/ugv0/telehandler/action/MoveToPosition', MoveToPositionAction)
    _action_goal.wait_for_server()

    pass_samples = 0
    total_samples = 0

    # List to keep all error values for CSV
    errors_list = []

    rospy.loginfo("Ready to start KPI validation test.")
    output_csv = "/home/telehandler_0/fortis_ws/scripts/Fortis_KPI/KPI/TE02_018/kpi_results.csv"

    for idx, desired_position in enumerate(desired_positions, start=1):
        input(f"Press Enter to send next goal to position: {desired_position}")
        rospy.loginfo(f"Sending goal to position: {desired_position}")

        goal = MoveToPositionGoal()
        goal.target_pose.header.frame_id = "base_link"
        goal.target_pose.pose.position.x = desired_position[0]
        goal.target_pose.pose.position.y = desired_position[1]
        goal.target_pose.pose.position.z = desired_position[2]
        goal.target_pose.pose.orientation.w = 1.0

        _action_goal.send_goal(goal)
        _action_goal.wait_for_result()

        current_pose = move_group.get_current_pose().pose
        actual_position = [current_pose.position.x,
                           current_pose.position.y, current_pose.position.z]

        error = euclidean_distance(actual_position, desired_position)
        # error /= 10 

        # Record error and pass/fail status
        passed = error <= 0.15
        if passed:
            pass_samples += 1
            rospy.loginfo(f"KPI Pass: Position error {error:.4f} m")
        else:
            rospy.logwarn(f"KPI Fail: Position error {error:.4f} m")

        total_samples += 1

        # Append test result (test number, error value, pass/fail)
        errors_list.append([idx, error, "PASS" if passed else "FAIL"])

    success_rate = (pass_samples / total_samples) * \
        100 if total_samples > 0 else 0

    rospy.loginfo("KPI Validation completed.")
    rospy.loginfo(f"Total tests: {total_samples}")
    rospy.loginfo(f"Passed tests: {pass_samples}")
    rospy.loginfo(f"Success rate: {success_rate:.2f}%")

    # Write all errors and summary to CSV
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)

        # Write header for detailed results
        writer.writerow(['Test Number', 'Position Error (m)', 'Result'])

        # Write all errors per test
        writer.writerows(errors_list)

        # Blank line separator between detailed errors and summary (optional)
        writer.writerow([])

        # Write summary stats
        writer.writerow(['Passed tests', 'Total tests', 'Success rate (%)'])
        writer.writerow([pass_samples, total_samples, f"{success_rate:.2f}"])

In [13]:
if __name__ == '__main__':
    try:
        rospy.init_node('move_group_commander_example', anonymous=True, disable_signals=True, log_level=rospy.INFO)
    except:
        pass    
    roscpp_initialize(sys.argv)
    try:
        main()
    except rospy.ROSInterruptException:
        pass
    except Exception as e:
        rospy.logerr(f"An error occurred: {e}")
        sys.exit(1)
    finally:
        rospy.on_shutdown(roscpp_shutdown)
        print("ROS shutdown complete.")
        sys.exit(0)

[INFO] [/move_group_commander_example_4060_1753956078155] [main] [50]: Initializing MoveIt Commander...
[ INFO] [/move_group_commander_wrappers_1753956078465122382] [planning_interface::MoveGroupInterface::MoveGroupInterfaceImpl::MoveGroupInterfaceImpl] [193]: Ready to take commands for planning group Arm_Group.
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [80]: Ready to start KPI validation test.
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [85]: Sending goal to position: [7.3, 0.0, 1.5]


[WARN] [/move_group_commander_example_4060_1753956078155] [main] [110]: KPI Fail: Position error 0.1614 m


[INFO] [/move_group_commander_example_4060_1753956078155] [main] [85]: Sending goal to position: [8.0, 0.0, 2.4]


[WARN] [/move_group_commander_example_4060_1753956078155] [main] [110]: KPI Fail: Position error 0.2557 m


[INFO] [/move_group_commander_example_4060_1753956078155] [main] [85]: Sending goal to position: [9.0, -2.0, 3.6]


...

[INFO] [/move_group_commander_example_4060_1753956078155] [main] [85]: Sending goal to position: [8.0, -1.5, 2.4]


[WARN] [/move_group_commander_example_4060_1753956078155] [main] [110]: KPI Fail: Position error 0.3157 m


[INFO] [/move_group_commander_example_4060_1753956078155] [main] [85]: Sending goal to position: [7.3, -1.0, 1.5]
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [120]: KPI Validation completed.
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [121]: Total tests: 5
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [122]: Passed tests: 0
[INFO] [/move_group_commander_example_4060_1753956078155] [main] [123]: Success rate: 0.00%
ROS shutdown complete.


[WARN] [/move_group_commander_example_4060_1753956078155] [main] [110]: KPI Fail: Position error 3.8111 m


SystemExit: 0